# Variational Autoencoders

This notebook accompanies the **ML Viz** lesson on VAEs.
We'll implement a VAE with the reparameterization trick and explore
how it structures the latent space.

**Companion lesson:** https://ml-viz.vercel.app/courses/generative-models/03-variational-autoencoders

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The Reparameterization Trick

In a VAE, the encoder outputs parameters of a distribution $q(z|x) = \mathcal{N}(\mu, \sigma^2)$.
To sample $z$ while keeping the operation differentiable:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Let's visualize this.

In [ ]:
np.random.seed(42)

mu, sigma = 2.0, 0.8
n_samples = 500

# Without reparameterization (non-differentiable)
z_wrong = np.random.normal(mu, sigma, n_samples)

# With reparameterization (differentiable)
eps = np.random.normal(0, 1, n_samples)
z_right = mu + sigma * eps

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Reparameterization Trick', color='white', fontsize=13, y=1.02)

# Epsilon distribution
axes[0].hist(eps, bins=30, color='#818cf8', alpha=0.8, edgecolor='#1a1d27')
axes[0].set_title('$\epsilon \sim \mathcal{N}(0, 1)$', color='white', fontsize=12)
axes[0].set_xlabel('$\epsilon$')

# Transformed distribution
axes[1].hist(z_right, bins=30, color='#14b8a6', alpha=0.8, edgecolor='#1a1d27')
x_range = np.linspace(-2, 5, 100)
axes[1].plot(x_range, norm.pdf(x_range, mu, sigma) * n_samples * 0.4, '--', color='white', alpha=0.5)
axes[1].set_title(f'$z = \mu + \sigma \cdot \epsilon$ ($\mu$={mu}, $\sigma$={sigma})', color='white', fontsize=12)
axes[1].set_xlabel('$z$')

# Show gradients flow
axes[2].text(0.5, 0.7, '$\epsilon$ (random)', fontsize=14, ha='center', color='#f43f5e', transform=axes[2].transAxes)
axes[2].text(0.5, 0.5, '$\\downarrow$', fontsize=20, ha='center', color='white', transform=axes[2].transAxes)
axes[2].text(0.5, 0.3, '$z = \mu + \sigma \cdot \epsilon$', fontsize=14, ha='center', color='#818cf8', transform=axes[2].transAxes)
axes[2].text(0.5, 0.1, 'Gradients flow through $\mu$ and $\sigma$ ✓', fontsize=11, ha='center', color='#14b8a6', transform=axes[2].transAxes)
axes[2].axis('off')
axes[2].set_title('Gradient Flow', color='white', fontsize=12)

plt.tight_layout()
plt.show()

## KL Divergence

The KL divergence between $q(z|x) = \mathcal{N}(\mu, \sigma^2)$ and the prior $p(z) = \mathcal{N}(0, 1)$ is:

$$D_{KL} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

Let's visualize how this changes with different $\mu$ and $\sigma$.

In [ ]:
mu_vals = np.linspace(-3, 3, 100)
sigma_vals = np.linspace(0.1, 3, 100)
MU, SIGMA = np.meshgrid(mu_vals, sigma_vals)

# KL divergence for a single dimension
KL = -0.5 * (1 + np.log(SIGMA**2) - MU**2 - SIGMA**2)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(MU, SIGMA, KL, levels=30, cmap='RdYlGn_r')
plt.colorbar(contour, ax=ax, label='$D_{KL}$')
ax.plot(0, 1, 'o', color='white', markersize=10, label='Prior: $\mu=0, \sigma=1$')
ax.set_xlabel('$\mu$')
ax.set_ylabel('$\sigma$')
ax.set_title('KL Divergence from $\mathcal{N}(0, 1)$', color='white', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print('KL = 0 at (mu=0, sigma=1) — exactly matches the prior.')
print('Moving away increases KL, penalizing the encoder.')

## Implementing a VAE

We'll build a simple VAE using NumPy to understand every detail.

In [ ]:
class VAE:
    def __init__(self, input_dim, latent_dim):
        # Encoder: input -> hidden -> (mu, logvar)
        scale1 = np.sqrt(2.0 / input_dim)
        self.W1 = np.random.randn(input_dim, 64) * scale1
        self.b1 = np.zeros(64)
        self.W_mu = np.random.randn(64, latent_dim) * 0.1
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(64, latent_dim) * 0.1
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder: latent -> hidden -> output
        scale2 = np.sqrt(2.0 / latent_dim)
        self.W3 = np.random.randn(latent_dim, 64) * scale2
        self.b3 = np.zeros(64)
        self.W4 = np.random.randn(64, input_dim) * np.sqrt(2.0 / 64)
        self.b4 = np.zeros(input_dim)
    
    def relu(self, x): return np.maximum(0, x)
    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -10, 10)))
    
    def encode(self, x):
        self.h1 = self.relu(x @ self.W1 + self.b1)
        self.mu = self.h1 @ self.W_mu + self.b_mu
        self.logvar = self.h1 @ self.W_logvar + self.b_logvar
        return self.mu, self.logvar
    
    def reparameterize(self, mu, logvar):
        std = np.exp(0.5 * logvar)
        eps = np.random.randn(*mu.shape)
        return mu + eps * std
    
    def decode(self, z):
        self.h3 = self.relu(z @ self.W3 + self.b3)
        return self.h3 @ self.W4 + self.b4
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar
    
    def sample(self, n):
        z = np.random.randn(n, self.W_mu.shape[1])
        return self.decode(z)

print('VAE class defined.')

## Train on 2D toy data

We'll use a simple 2D dataset — a circle — and train a VAE to learn its structure.

In [ ]:
np.random.seed(42)
n = 300
theta = np.random.uniform(0, 2 * np.pi, n)
r = 1.0 + 0.1 * np.random.randn(n)
X = np.column_stack([r * np.cos(theta), r * np.sin(theta)])

vae = VAE(input_dim=2, latent_dim=2)

losses = []
for epoch in range(2000):
    # Forward
    x_hat, mu, logvar = vae.forward(X)
    
    # Loss = reconstruction + KL
    recon = np.mean((X - x_hat) ** 2)
    kl = -0.5 * np.mean(1 + logvar - mu**2 - np.exp(logvar))
    loss = recon + kl
    losses.append(loss)
    
    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1:4d} | Recon: {recon:.4f} | KL: {kl:.4f} | Total: {loss:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#818cf8', linewidth=1)
ax.set_title('VAE Training Loss', color='white', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
plt.tight_layout()
plt.show()

## Latent space and generation

The VAE forces the latent space to be smooth and structured.
We can now sample from $\mathcal{N}(0, I)$ and decode to generate new data.

In [ ]:
z_encoded = vae.encode(X)[0]
x_reconstructed = vae.forward(X)[0]
x_generated = vae.sample(300)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
fig.suptitle('VAE Results', color='white', fontsize=13, y=1.02)

axes[0].scatter(X[:, 0], X[:, 1], c='#94a3b8', s=15, alpha=0.6)
axes[0].set_title('Original Data', color='white', fontsize=11)

axes[1].scatter(z_encoded[:, 0], z_encoded[:, 1], c=t[:300] if len(t := theta) > 300 else theta, cmap='viridis', s=15, alpha=0.7)
axes[1].set_title('Latent Space', color='white', fontsize=11)
axes[1].set_xlabel('$z_1$')
axes[1].set_ylabel('$z_2$')

axes[2].scatter(x_reconstructed[:, 0], x_reconstructed[:, 1], c='#14b8a6', s=15, alpha=0.6)
axes[2].set_title('Reconstructions', color='white', fontsize=11)

axes[3].scatter(x_generated[:, 0], x_generated[:, 1], c='#eab308', s=15, alpha=0.6)
axes[3].set_title('Generated (sample from $\mathcal{N}(0,I)$)', color='white', fontsize=11)

for ax in axes:
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## Latent space manifold

Let's visualize what different regions of the latent space decode to,
by sampling on a grid.

In [ ]:
n_grid = 12
z1 = np.linspace(-2, 2, n_grid)
z2 = np.linspace(-2, 2, n_grid)

fig, axes = plt.subplots(n_grid, n_grid, figsize=(12, 12))
fig.suptitle('Latent Space Manifold — What Each Region Generates', color='white', fontsize=14, y=1.01)

for i, z_val_1 in enumerate(z1):
    for j, z_val_2 in enumerate(z2):
        z = np.array([[z_val_1, z_val_2]])
        x = vae.decode(z)
        ax = axes[n_grid - 1 - i, j]
        # Draw a small circle at the decoded position
        circle = plt.Circle(x[0], 0.15, color='#818cf8', alpha=0.8)
        ax.add_patch(circle)
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_aspect('equal')
        ax.axis('off')

plt.tight_layout()
plt.show()